In [146]:
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_sm")

df = pd.read_csv("datathon_year_3.csv")

# rows, cols
print(df.shape)
# types
print(df.dtypes)

(3995, 15)
Idea #                                int64
Original Title                          str
Challenge                               str
Solution                                str
Audience                                str
Borough                                 str
Neighborhood(s)                         str
Host of Idea Generation Workshop        str
Impact Area                             str
Sub-Category                            str
Type of Idea                            str
Idea Zip Code                       float64
Idea Generation Session Zip Code    float64
Idea Source                             str
Insights                                str
dtype: object


In [147]:
# preview of first 2 rows
print(df.head(2))

   Idea #                          Original Title  \
0      14  Pathways to Success: Bilingual Support   
1     111                    Youth Launch Project   

                                           Challenge  \
0  Teens of immigrants struggle with the college ...   
1  A challenge in Harlem is helping students lear...   

                                            Solution    Audience    Borough  \
0  A mentorship program that would offer bilingua...  Immigrants     Queens   
1  A youth enterprenuership program that teaches ...       Youth  Manhattan   

  Neighborhood(s)                   Host of Idea Generation Workshop  \
0             NaN  [10/10/2024 @ 2:00 PM] DOE Civics For All Even...   
1          Harlem  [10/15/2024 @ 5:30 PM] Lee Heritage Media LLC ...   

  Impact Area                                       Sub-Category  \
0   Education  EDU - Adult Educational Programs,EDU - Aftersc...   
1   Education  EDU - Professional Skills Development,EDU - Jo...   

           

In [148]:
# drop unnecessary columns 
df = df.drop(['Host of Idea Generation Workshop', 'Idea Zip Code', 'Idea Generation Session Zip Code', 'Idea Source'], axis = 1)
print(df.dtypes)

Idea #             int64
Original Title       str
Challenge            str
Solution             str
Audience             str
Borough              str
Neighborhood(s)      str
Impact Area          str
Sub-Category         str
Type of Idea         str
Insights             str
dtype: object


In [149]:
# check for NA values
print(df.isna().sum())

Idea #               0
Original Title      41
Challenge            6
Solution            13
Audience             1
Borough              0
Neighborhood(s)    201
Impact Area          4
Sub-Category       474
Type of Idea       703
Insights           838
dtype: int64


In [ ]:
# Check rows where Challenge or Solution is duplicated
duplicated_rows = df[df.duplicated(subset=["Challenge"], keep=False) |
                     df.duplicated(subset=["Solution"], keep=False)]

print("Duplicated rows:\n", duplicated_rows)

# drop duplicates
df = df.drop_duplicates(subset=["Challenge"], keep='first')
df = df.drop_duplicates(subset=["Solution"], keep='first')

Duplicated rows:
 Empty DataFrame
Columns: [Idea #, Original Title, Challenge, Solution, Audience, Borough, Neighborhood(s), Impact Area, Sub-Category, Type of Idea, Insights]
Index: []
Duplicated rows:
 Empty DataFrame
Columns: [Idea #, Original Title, Challenge, Solution, Audience, Borough, Neighborhood(s), Impact Area, Sub-Category, Type of Idea, Insights]
Index: []


In [134]:
# stop_words = spacy.lang.en.stop_words.STOP_WORDS
# print('Number of stopwords: %d' % len(stop_words))
# print(list(stop_words))

In [135]:
# NaN Check
# If NaN left in data, it changes the column to a float causing errors
df["Challenge"].apply(type).value_counts()

# drop missing values
df = df.dropna(subset=["Challenge", "Solution"])

In [ ]:
# remove characters from rows that start with bad chars
df[['Challenge', 'Solution']] = df[['Challenge', 'Solution']].apply(
    lambda col: col.str.lstrip("\"'@-=")
)

# ^^^ above uses anon func lambda to apply the following func into each col, acts as wrapper so can use .apply
# equivalent to:
# def clean_start(col):
#     return col.str.lstrip("\"'@-=")

# df[cols_to_clean] = df[cols_to_clean].apply(clean_start)

# remove "=" from text
# for each selected column, replace characters in all strings
df[['Challenge', 'Solution']] = df[['Challenge', 'Solution']].apply(
    lambda col: col.str.replace("=", "", regex=False))

In [119]:
# noun chunk extraction
def extract_noun_chunk(text):
    doc = nlp(text.lower())
    return " | ".join([chunk.text for chunk in doc.noun_chunks])

df['Challenge_noun_chunk'] = df['Challenge'].apply(extract_noun_chunk)
df['Solution_noun_chunk'] = df['Solution'].apply(extract_noun_chunk)
temp_df_nc = df[['Challenge_noun_chunk', 'Solution_noun_chunk']]
print(temp_df_nc.head(10))

                                Challenge_noun_chunk  \
0  teens | immigrants | the college application p...   
1  a challenge | harlem | students | their own bu...   
2  some seniors | limited credit | no income | ho...   
3           many teens | methods | income | canarsie   
4  many younger parents | parents | no help | par...   
5  schools | a rigid structure | that | almost no...   
6  no clubs | elementary school grades | high sch...   
7  pakistani women immigrants | no support form p...   
8  lack | “relevant” work opportunities | teens/y...   
9                                      people | they   

                                 Solution_noun_chunk  
0  a mentorship program | that | bilingual worksh...  
1  a youth enterprenuership program | that | stud...  
2  an urgent need | senior apartments | queens | ...  
3  a program | teens | available resources | empl...  
4  more parenting coaching classes | young adults...  
5  programs | that | host clubs | students | know... 

In [120]:
# lemmatize function
def clean_lemmatize(text):
    doc = nlp(text.lower())
    return " ".join([
        token.lemma_
        for token in doc
        # punctuation   
        if not token.is_punct   
        # spaces 
        and not token.is_space   
        # stopwords
        and not token.is_stop  
    ])

df['Challenge_lem'] = df['Challenge'].apply(clean_lemmatize)
df['Solution_lem'] = df['Solution'].apply(clean_lemmatize)
lem_df = df[['Challenge_lem', 'Solution_lem']]
print(lem_df.head(20))

                                        Challenge_lem  \
0   teen immigrant struggle college application pr...   
1   challenge harlem help student learn start busi...   
2   senior limited credit income find housing majo...   
3                         teen method income canarsie   
4                young parent parent help grow parent   
5   school rigid structure leave time extra curric...   
6   club elementary school grade like high school ...   
7   pakistani woman immigrant unemployed support f...   
8   lack relevant work opportunity teen youth word...   
9                              people drown know swim   
10  lack access free financial literacy education ...   
11  new york city housing crisis landlord advantag...   
12                youth outlet express gift misdirect   
13                              people cook mean feed   
14  folk limited access housing struggle food inse...   
15  youth employment trade programming outreach of...   
16  awareness autism student di

In [121]:
# dependency parsing

# def show_dependencies(text):
#     doc = nlp(text)
#     for token in doc:
#         print(token.text, token.dep_, token.head.text)

# subject object verb extraction, including prepositional objs
def extract_svo(text):
    doc = nlp(text)
    svos = []
    
    for token in doc:
        if token.pos_ == "VERB":
            subjects = [child for child in token.children if child.dep_ in ["nsubj", "nsubjpass"]]
            objects = [child for child in token.children if child.dep_ in ["dobj", "attr", "oprd"]]
            
            # include prepositional objects
            for prep in [child for child in token.children if child.dep_ == "prep"]:
                objects.extend([child for child in prep.children if child.dep_ == "pobj"])
            
            for subj in subjects:
                for obj in objects:
                    svos.append((subj.text, token.lemma_, obj.text))
    
    return svos

df["svos_problems"] = df["Challenge"].apply(extract_svo)
df["svos_solutions"] = df["Solution"].apply(extract_svo)

svo_df = df[['svos_problems', 'svos_solutions']]
print(svo_df.head(20))

# test for checking visualization of SVO on row 5 (cant use on whole column)
# for text in df["Community problems"].head(5):
#     doc = nlp(text)
#     displacy.render(doc, style="dep", jupyter=True)

                                        svos_problems  \
0   [(Teens, struggle, processes), (parents, advis...   
1                                                  []   
2                                                  []   
3                            [(teens, have, methods)]   
4                          [(parents, grow, parents)]   
5   [(Schools, have, structure), (that, leave, tim...   
6   [(This, offer, experiences), (This, offer, age...   
7                                                  []   
8                                                  []   
9                                                  []   
10                                                 []   
11  [(that, take, advantage), (that, take, people)...   
12                           [(youth, have, outlets)]   
13                                                 []   
14                                                 []   
15              [(employment, oftentime, experience)]   
16             [(child, have, I

In [ ]:
# most common issues from challenge lem
from collections import Counter

all_words = " ".join(df["Challenge_lem"]).split()
word_counts = Counter(all_words).most_common(30)
df_counts = pd.DataFrame(word_counts, columns=["Common Issue", "Count"])
print(df_counts)

   Common Issue  Count
0             =   1867
1        people   1457
2     community    983
3          need    848
4         youth    632
5          lack    606
6        school    545
7       program    543
8          help    519
9       housing    466
10          job    351
11        child    349
12       health    335
13    immigrant    326
14        young    299
15        adult    296
16    challenge    295
17          low    293
18       mental    292
19      support    290
20      problem    285
21       access    281
22       income    274
23     resource    274
24       senior    273
25         food    273
26       parent    272
27      english    268
28       street    266
29         like    255


In [127]:
# most common issues from noun chunk challenges
all_chunks = []
for chunks in df["Challenge_noun_chunk"]:
    all_chunks.extend(chunks.split(" | "))

Counter(all_chunks).most_common(30)

[('they', 788),
 ('people', 719),
 ('that', 637),
 ('i', 580),
 ('we', 515),
 ('it', 441),
 ('them', 381),
 ('who', 355),
 ('lack', 332),
 ('youth', 234),
 ('the community', 213),
 ('a lot', 189),
 ('immigrants', 166),
 ('school', 159),
 ('which', 147),
 ('children', 144),
 ('the youth', 142),
 ('disabilities', 136),
 ('access', 135),
 ('parents', 130),
 ('housing', 130),
 ('seniors', 130),
 ('this', 125),
 ('resources', 121),
 ('help', 115),
 ('the streets', 114),
 ('=', 114),
 ('young people', 111),
 ('english', 109),
 ('education', 105)]

In [ ]:
# output
# in-notebook display
df

# csv export
# df.to_csv("output.csv", index=False)

,Idea #,Original Title,Challenge,Solution,Audience,Borough,Neighborhood(s),Impact Area,Sub-Category,Type of Idea,Insights,Challenge_noun_chunk,Solution_noun_chunk,Challenge_lem,Solution_lem,svos_problems,svos_solutions
0,14,Pathways to Success: Bilingual Support,Teens of immigrants struggle with the college ...,A mentorship program that would offer bilingua...,Immigrants,Queens,NaN,Education,"EDU - Adult Educational Programs,EDU - Aftersc...","""Classes, Workshop or Training""","Problem,Solution",teens | immigrants | the college application p...,a mentorship program | that | bilingual worksh...,teen immigrant struggle college application pr...,mentorship program offer bilingual workshop se...,"[(Teens, struggle, processes), (parents, advis...","[(that, offer, workshops), (We, train, mentors..."
1,111,Youth Launch Project,A challenge in Harlem is helping students lear...,A youth enterprenuership program that teaches ...,Youth,Manhattan,Harlem,Education,"EDU - Professional Skills Development,EDU - Jo...","""Classes, Workshop or Training""",Solution,a challenge | harlem | students | their own bu...,a youth enterprenuership program | that | stud...,challenge harlem help student learn start busi...,youth enterprenuership program teach student s...,[],"[(that, teach, students)]"
2,209,Housing workshop for seniors,For some seniors with limited credit and no in...,"There is an urgent need for senior apartments,...","Older Adults,Unhoused People,Low Income People",Queens,Queens (Flushing & Bayside),"Other,Social Services & Accessiblity",SS - Housing Assistance,Service Delivery,"Problem,Solution",some seniors | limited credit | no income | ho...,an urgent need | senior apartments | queens | ...,senior limited credit income find housing majo...,urgent need senior apartment particularly quee...,[],[]
3,289,Teen Employee Connect (TEC),Many teens do not have methods of income in Ca...,develop a program to connect teens with availa...,"Youth,Low Income People",Brooklyn,"Canarsie, Flatbush",Workforce Development,"EDU - Professional Skills Development,WD - Job...","""Classes, Workshop or Training""",Solution,many teens | methods | income | canarsie,a program | teens | available resources | empl...,teen method income canarsie,develop program connect teen available resourc...,"[(teens, have, methods)]",[]
4,348,"""Your not alone""",Many younger parents and parents with no help ...,More parenting coaching classes for young adul...,Parents,Brooklyn,"Brownsville, Brooklyn",Social Services & Accessiblity,SS - Parenting Support & Prevention,"""Classes, Workshop or Training""","Problem,Solution",many younger parents | parents | no help | par...,more parenting coaching classes | young adults...,young parent parent help grow parent,parenting coaching class young adult support h...,"[(parents, grow, parents)]","[(parents, gain, knowledge), (parents, gain, t..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3990,4334,Bridging the Exponentially Growing Digital Div...,Technology often advances at breakneck speeds ...,This project proposes a multifaceted approach ...,"Youth,Older Adults,Public Housing Residents,Ju...",Bronx,All areas.,Education,NaN,NaN,NaN,technology | breakneck speeds | this | the rea...,this project | a multifaceted approach | the a...,technology advance breakneck speed especially ...,project propose multifaceted approach bridge a...,"[(Technology, advance, speeds), (which, releas...","[(project, propose, approach), (We, foster, pa..."
3991,4509,Auxiliary Police Recruitment,Most people are opting to work from home and s...,"Hire 10 Auxiliary police, put them through a s...",Someone Else,Manhattan,This should be for major MTA depos in Brooklyn...,Public Safety,PS - Community Safety,Service Delivery,"Problem,Solution",most people | home | they | safe commuting | t...,10 auxiliary police | them | a safety voluntee...,people opt work home shop online feel safe com...,hire 10 auxiliary police safety volunteer trai...,"